In [ ]:
import os
os.environ["HF_HOME"] = "/projectnb/vkolagrp/skowshik/.cache/"


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"device: {device}")

In [ ]:
model_id = "Qwen/Qwen-2.5-3B-Instruct"
n_devices = 1

In [ ]:
# load model
model = AutoModelForCausalLM.from_pretrained(
    model_id, 
    cache_dir = "/projectnb/vkolagrp/skowshik/.cache/",
    torch_dtype="auto",
    device_map="auto")

# load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id)

In [ ]:

# define a hook function that caches activations
def cache_hook(cache):
	def hook(module, input, output):
		cache.append(output[0]) # the output of the residual stream is actually a tuple, where the first entry is the activation
	return hook



In [ ]:
# Loop over all layers and attention heads in the model
num_layers = len(model.transformer.h)
# Try to infer the number of heads by inspecting a typical transformer block
example_block = model.transformer.h[0]
if hasattr(example_block.self_attn, 'num_heads'):
    num_heads = example_block.self_attn.num_heads
elif hasattr(example_block.self_attn, 'num_attention_heads'):
    num_heads = example_block.self_attn.num_attention_heads
else:
    # As a fallback, inspect the attention projections
    # (e.g., use q_proj weight shape [embed_dim, hidden_dim])
    # For Qwen-style, let's try:
    num_heads = model.config.num_attention_heads if hasattr(model.config, "num_attention_heads") else None

print(f"Total layers: {num_layers}")
print(f"Heads per layer: {num_heads}")

for layer_idx in range(num_layers):
    block = model.transformer.h[layer_idx]
    print(f"\nLayer {layer_idx}:")
    for head_idx in range(num_heads):
        print(f"  Head {head_idx}")
        # At this point, you can hook attention head activations, query params, etc., 
        # e.g. block.self_attn; details depend on where/what you wish to analyze


In [ ]:
# define layer to do the activation steering on
layer_id = 5

# get internal activations
cache = []
handle = model.transformer.h[layer_id].register_forward_hook(cache_hook(cache))
inputs = tokenizer("Love", return_tensors="pt").to(device)
_ = model(**inputs)
inputs = tokenizer("Hate", return_tensors="pt").to(device)
_ = model(**inputs)
handle.remove()  # it's very important to keep track of hook handles and remove the hooks 
act_love = cache[0]
act_hate = cache[1]

print(f"act_love.shape: {act_love.shape}")
print(f"act_hate.shape: {act_hate.shape}")

